# Study 898 — Managed-Vol Equity 🎚️

**Turn SPY into a thermostat: aim for a constant ~12% volatility — lean in when the market
is calm, step out when it is stormy. Does that raise the Sharpe, or just tame the ride?**

Moreira & Muir (2017) showed that scaling equity *inversely to recent volatility* raises the
risk-adjusted return. We take the self-contained single-asset version on **SPY vs bills
(BIL)**: hold `w = min(2.0, 12% / RV_21d)` of SPY, the rest in real T-bills, one execution
lag. Sample 2007-05-30 → 2026-06-30 (4,802 daily total-return closes). Every race
is **excess-of-cash on both legs**.

*Numbers below are the frozen headline (`docs/results.md`, fingerprint `f7ef586be44c`);
the live cells run the fast synthetic control. Single ~19-year SPY tape — short-history.*


## 1. The idea in one picture

Buy-and-hold SPY lets its volatility roam — calm 10% one year, a terrifying 40% in a crash. A **vol thermostat** rescales daily so the *portfolio* vol stays near a set point (12%): when SPY gets loud, you hold less of it and more cash; when it is quiet, you hold a bit more (up to 2×). The pitch is that you shed risk you were not being paid to bear — a smoother ride, maybe a better Sharpe.

In [1]:
R = {'start': '2007-05-30', 'end': '2026-06-30', 'n_rows': 4802, 'n_days': 4780, 'fingerprint': 'f7ef586be44c', 'sh_strat': 0.659, 'vol_strat': 13.4, 'dd_strat': -31.1, 'cagr_strat': 8.26, 'wealth_strat': 4.51, 'sh_bh': 0.549, 'vol_bh': 19.9, 'dd_bh': -56.5, 'cagr_bh': 9.35, 'wealth_bh': 5.45, 'sharpe_gap': 0.11, 'alpha': 2.78, 't_alpha': 1.67, 'beta': 0.56, 'appraisal': 0.36, 'avg_w': 0.96, 'share_lev': 41.5, 'turnover': 9.15, 'exposure_bps': 2.407, 'timing_bps': 1.102, 'diff_bps': -0.822, 't_diff': -0.97, 'boot_gap': 0.11, 'boot_lo': -0.128, 'boot_hi': 0.355, 'boot_pneg': 0.195, 'vol_median': 12.5, 'vol_p10': 8.8, 'vol_p90': 17.4, 'vol_band': 86.0, 'bh_vol_p90': 27.8, 'era_early_t': 0.96, 'era_early_sh': 0.449, 'era_early_shbh': 0.347, 'era_early_n': 2143, 'era_late_t': 1.08, 'era_late_sh': 0.827, 'era_late_shbh': 0.759, 'era_late_n': 2637, 'cost1_sh': 0.652, 'cost1_alpha': 2.69, 'cost5_sh': 0.613, 'cost5_alpha': 2.16, 'cost5_dd': -31.4, 'pl_alpha_obs': 2.78, 'pl_alpha_mean': -0.11, 'pl_alpha_sd': 2.03, 'pl_p_alpha': 0.07, 'pl_gap_obs': 0.11, 'pl_gap_mean': -0.055, 'pl_p_gap': 0.05, 'pl_dd_obs': -31.1, 'pl_dd_mean': -57.4, 'pl_p_dd': 0.0, 'syn_null_t': -0.01, 'syn_null_fire': 7, 'syn_planted_t': 4.98, 'syn_planted_alpha': 9.4, 'syn_planted_fire': 97, 'crash': [('GFC 2008-09', -30.6, -56.5), ('2018 Q4', -18.0, -19.8), ('COVID 2020', -13.7, -33.9), ('2022 bear', -15.2, -25.0)]}

In [2]:
print('MANAGED  vs  BUY & HOLD SPY  (excess of cash, 2007-2026)')
print(f"  Sharpe   {R['sh_strat']:.3f}   vs   {R['sh_bh']:.3f}   (+{R['sharpe_gap']:.3f})")
print(f"  ann vol  {R['vol_strat']:.1f}%   vs   {R['vol_bh']:.1f}%")
print(f"  max DD  {R['dd_strat']:.1f}%   vs  {R['dd_bh']:.1f}%   <- the heart-attack cut")

MANAGED  vs  BUY & HOLD SPY  (excess of cash, 2007-2026)
  Sharpe   0.659   vs   0.549   (+0.110)
  ann vol  13.4%   vs   19.9%
  max DD  -31.1%   vs  -56.5%   <- the heart-attack cut


## 2. The heart-attack ledger — where the shield shows up

The drawdown cut concentrates exactly in the crashes, and it is genuine *timing*: a 200-seed placebo that shuffles the vol signal (same weight menu, wrong days) averages a **−57.4%** drawdown — none of 200 shuffles matches the real −31.1% (**p = 0.000**). Cutting exposure *on the right days* is doing the work.

In [3]:
for name, s, b in R['crash']:
    print(f"{name:<12}  managed {s:+.1f}%   buy&hold {b:+.1f}%")
print(f"\nfull sample   managed {R['dd_strat']:+.1f}%   buy&hold {R['dd_bh']:+.1f}%")
print(f"placebo (shuffled vol, 200 seeds): mean DD {R['pl_dd_mean']:+.1f}%  ->  p = {R['pl_p_dd']:.3f}")

GFC 2008-09   managed -30.6%   buy&hold -56.5%
2018 Q4       managed -18.0%   buy&hold -19.8%
COVID 2020    managed -13.7%   buy&hold -33.9%
2022 bear     managed -15.2%   buy&hold -25.0%

full sample   managed -31.1%   buy&hold -56.5%
placebo (shuffled vol, 200 seeds): mean DD -57.4%  ->  p = 0.000


## 3. Is the sort just lucky? A live synthetic control

We plant a vol-return *disconnect* in a seeded toy world (`disconnect=2`, mean falls as variance rises) and check the detector recovers the timing alpha — and stays *silent* on the null (`disconnect=0`, risk fully priced). No network.

In [4]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from managed_vol import strategy as st
# single-seed alpha t is noisy (sd ~1.3), so average a handful of seeds
nulls = np.array([st.synthetic_detect(0.0, seed=898+s, n_days=4000)['t_alpha'] for s in range(6)])
plant = np.array([st.synthetic_detect(2.0, seed=898+s, n_days=4000)['t_alpha'] for s in range(6)])
print(f"null world   : mean alpha t = {nulls.mean():+.2f}  (should be ~0)")
print(f"planted world: mean alpha t = {plant.mean():+.2f}  (should light up)")

null world   : mean alpha t = +0.28  (should be ~0)
planted world: mean alpha t = +4.51  (should light up)


## 4. The honest verdict — a real shield, an unproven edge

On the real SPY tape the thermostat **halves the heart attacks** (max DD **-31.1% vs -56.5%**) and holds vol near 12% (median **12.5%**) — that half is real and robust. But the celebrated **higher-Sharpe** half never certifies: the +0.110 Sharpe gain has a bootstrap CI of **[-0.128, +0.355]** that straddles zero, and the timing alpha is **+2.78%/yr at t = 1.67** (below the *t* ≥ 2 bar). And because the book runs β = 0.56 < 1 you *give up* ~1.1 pp/yr of excess return for the smoother path. **Signal: Mixed** (real tail control, weak Sharpe), **Tradability: Fragile** (cheap to run, but a risk shield, not a bankable edge).